# Dynamic PET quenching of a tethered dye

The decay of a dye on a flexible linker, from the structure alone. QuEst builds
the dye's accessible volume, diffuses it there, and quenches it by
photo-induced electron transfer whenever it touches an electron-rich side
chain.

This replaces the old `mfm`-based notebooks: same science, current API.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "quest_notebook.py").exists() else Path.cwd() / "notebooks"))
from quest_notebook import REPO, STRUCTURE, quick_project, fret_project

import numpy as np
import matplotlib.pyplot as plt

print("structure:", STRUCTURE.name)

In [ ]:
from quest.api import simulate_project, top_quenchers

result = simulate_project(quick_project(residue=100), save_outputs=False)
print(f"quantum yield   {result.quantum_yield_donor:.4f}")
print(f"mean lifetime   {result.lifetime_donor:.3f} ns   (tau0 = 4.2 ns)")
print(f"contact         {result.collisions_fraction * 100:.1f} % of frames")


## Which residues do the quenching

Every run reports the contribution per residue type. Residue 100 sits next to a
tryptophan, and the attribution says so.

In [ ]:
for entry in top_quenchers(result.aa_residence, limit=5):
    print(f"{entry['residue_name']:>4s}  {entry['quench_rate_fraction'] * 100:5.1f} % of the quenching")

## The decay is not single-exponential

The dye has one unquenched lifetime, but it samples environments that quench it
differently — so the observable decay is a superposition.

In [ ]:
quenched = simulate_project(quick_project(100), save_outputs=False)
free = simulate_project(quick_project(15), save_outputs=False)

fig, ax = plt.subplots(figsize=(6, 4))
for label, r in (("residue 100 (next to TRP)", quenched), ("residue 15 (little quenching)", free)):
    counts = np.clip(r.donor_counts, 1e-1, None)
    ax.semilogy(r.time, counts / counts.max(), label=f"{label}: QY {r.quantum_yield_donor:.2f}")
ax.set_xlabel("time [ns]"); ax.set_ylabel("normalised counts")
ax.set_xlim(0, 25); ax.legend(); ax.set_title("Donor decay at two labelling sites")
plt.tight_layout()
